# 🌽 DenseNet121 — Corn Leaf Disease Detection Trainer (TensorFlow / Keras)
> **Framework:** TensorFlow 2.x / Keras 3 | **Model:** DenseNet121 (Transfer Learning + Fine-tuning)  
> **Dataset:** Corn Leaf Disease (PlantVillage atau custom) | **Task:** Multi-class Classification

---
### 📋 Daftar Cell
| # | Cell | Keterangan |
|---|------|-----------|
| 1 | Cek Lingkungan | Verifikasi GPU & versi TensorFlow/Keras |
| 2 | Install Library | Pastikan dependency terinstal |
| 3 | Import | Import semua modul |
| 4 | ⚙️ Konfigurasi Dataset | **Ubah di sini** sebelum mulai |
| 5 | Load Dataset Kaggle | Jalankan jika sumber = Kaggle |
| 6 | Mount Google Drive | Jalankan jika sumber = Drive |
| 7 | Siapkan Dataset | Split train/val/test otomatis |
| 8 | Eksplorasi Dataset | Statistik & sampel gambar |
| 9 | ⚙️ Konfigurasi Hyperparameter | **Ubah di sini** sesuai kebutuhan |
| 10 | tf.data Pipeline | Dataset + augmentasi |
| 11 | Bangun Model | DenseNet121 custom head (Functional API) |
| 12 | Utilitas Training | Checkpoint & helper functions |
| 13 | Loop Training | Training dengan live logs |
| 14 | Mulai / Lanjutkan Training | **Jalankan untuk training** |
| 15 | Visualisasi History | Kurva loss & accuracy |
| 16 | Evaluasi Model | Confusion matrix & report |
| 17 | Simpan Model Akhir | Export ke .keras / TFLite |
| 18 | ⚙️ Konfigurasi Inferensi | **Ubah di sini** sebelum inferensi |
| 19 | Inferensi Gambar Baru | Deteksi penyakit pada gambar baru |

> 💡 **Catatan:** Notebook ini adalah versi TensorFlow/Keras dari trainer PyTorch. Struktur cell & alur kerja identik agar mudah dibandingkan, tapi seluruh implementasi menggunakan `tf.keras` (Keras 3) dengan API terbaru: `image_dataset_from_directory`, `tf.data`, preprocessing layers, dan format model `.keras`.

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 1 — Cek Lingkungan & GPU
# ═══════════════════════════════════════════════════
import sys
import tensorflow as tf
from tensorflow import keras

print("=" * 60)
print("🖥️  INFORMASI LINGKUNGAN")
print("=" * 60)
print(f"Python       : {sys.version.split()[0]}")
print(f"TensorFlow   : {tf.__version__}")
print(f"Keras        : {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

gpus = tf.config.list_physical_devices('GPU')
print(f"GPU tersedia : {len(gpus) > 0}")

if gpus:
    for i, gpu in enumerate(gpus):
        print(f"GPU [{i}]      : {gpu.name}")
    try:
        details = tf.config.experimental.get_device_details(gpus[0])
        if 'device_name' in details:
            print(f"GPU Model    : {details['device_name']}")
    except Exception:
        pass
    # Aktifkan memory growth agar tidak langsung menyedot semua VRAM
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
else:
    print()
    print("⚠️  WARNING: GPU tidak aktif!")
    print("   Aktifkan GPU: Runtime → Change runtime type → T4 GPU")
    print("   Training di CPU akan SANGAT lambat.")

print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2 — Install Library Tambahan
# ═══════════════════════════════════════════════════
# TensorFlow & Keras sudah tersedia di Colab secara default.
# Instal library pendukung yang belum tersedia:
%pip install -q kaggle scikit-learn seaborn matplotlib Pillow
print("✅ Semua library siap.")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3 — Import Semua Modul
# ═══════════════════════════════════════════════════
import os, sys, json, time, math, shutil, random, warnings
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

warnings.filterwarnings("ignore")


# ── Reproducibility ────────────────────────────────────────
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

AUTOTUNE = tf.data.AUTOTUNE
print(f"✅ Semua modul berhasil diimport.")
print(f"🔧 tf.data.AUTOTUNE siap digunakan untuk pipeline data.")

## ⚙️ KONFIGURASI DATASET
> **Atur variabel di bawah sebelum menjalankan cell dataset**

| Variabel | Keterangan |
|----------|-----------|
| `DATASET_SOURCE` | `'kaggle'` atau `'drive'` |
| `DRIVE_MODE` | `'split_folders'` (sudah ada train/test) atau `'single_folder'` (satu folder, akan di-split) |
| `SPLIT_TEST_RATIO` | Proporsi test set, misal `0.2` = 20% |
| `SPLIT_VALID_RATIO` | Proporsi val dari total dataset, misal `0.1` = 10% |

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 4 — Konfigurasi Dataset
# ═══════════════════════════════════════════════════
# ── Sumber Dataset ─────────────────────────────────
DATASET_SOURCE = 'drive'     # 'kaggle' | 'drive'

# ── Kaggle ─────────────────────────────────────────
KAGGLE_DATASET   = 'vipoooool/new-plant-diseases-dataset'
#                  ^^ Ganti dengan username/nama-dataset di Kaggle
KAGGLE_UNZIP_DIR = '/content/kaggle_dataset'

# ── Google Drive ───────────────────────────────────
DRIVE_MODE       = 'split_folders'  # 'split_folders' | 'single_folder'

# Jika DRIVE_MODE = 'split_folders':
DRIVE_TRAIN_DIR  = '/content/drive/MyDrive/datasets/corn/train'
DRIVE_TEST_DIR   = '/content/drive/MyDrive/datasets/corn/test'
DRIVE_VALID_DIR  = ''   # Kosongkan jika tidak ada folder validasi terpisah

# Jika DRIVE_MODE = 'single_folder':
DRIVE_SINGLE_DIR = '/content/drive/MyDrive/datasets/corn'

# ── Split Ratio (untuk 'single_folder' atau auto-split) ──
SPLIT_TEST_RATIO  = 0.20   # 20% → test
SPLIT_VALID_RATIO = 0.10   # 10% → validasi (dari total dataset)

# ── Output ─────────────────────────────────────────
OUTPUT_DIR      = '/content/output'
CHECKPOINT_DIR  = f'{OUTPUT_DIR}/checkpoints'
os.makedirs(OUTPUT_DIR,     exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("✅ Konfigurasi dataset tersimpan:")
print(f"   Sumber     : {DATASET_SOURCE.upper()}")
print(f"   Drive mode : {DRIVE_MODE}")
print(f"   Output dir : {OUTPUT_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 5 — Load Dataset dari Kaggle
# ═══════════════════════════════════════════════════
# ⚠️  Jalankan HANYA jika DATASET_SOURCE == 'kaggle'
if DATASET_SOURCE == 'kaggle':
    from google.colab import files

    print("📂 Upload file kaggle.json")
    print("   (Diperoleh dari: https://www.kaggle.com/settings → API → Create New Token)")
    uploaded = files.upload()

    kaggle_dir = Path('/root/.config/kaggle')
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / 'kaggle.json'

    if 'kaggle.json' in uploaded:
        kaggle_json.write_bytes(uploaded['kaggle.json'])
        kaggle_json.chmod(0o600)
        print("✅ kaggle.json dikonfigurasi.")
    else:
        raise FileNotFoundError("❌ File kaggle.json tidak ditemukan dalam upload!")

    os.makedirs(KAGGLE_UNZIP_DIR, exist_ok=True)
    print(f"\n📥 Mendownload: {KAGGLE_DATASET} ...")
    ret = os.system(f"kaggle datasets download -d {KAGGLE_DATASET} -p {KAGGLE_UNZIP_DIR} --unzip -q")
    if ret != 0:
        print("❌ Download gagal. Periksa nama dataset dan koneksi internet.")
    else:
        print(f"✅ Download selesai → {KAGGLE_UNZIP_DIR}")
        print("\n📁 Isi folder:")
        for p in sorted(Path(KAGGLE_UNZIP_DIR).iterdir()):
            print(f"   {'[DIR] ' if p.is_dir() else '      '}{p.name}")
else:
    print(f"ℹ️  DATASET_SOURCE = '{DATASET_SOURCE}' → cell ini dilewati.")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 6 — Mount Google Drive
# ═══════════════════════════════════════════════════
# ⚠️  Jalankan HANYA jika DATASET_SOURCE == 'drive'
if DATASET_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("✅ Google Drive berhasil di-mount.")

    if DRIVE_MODE == 'split_folders':
        for name, d in [('Train', DRIVE_TRAIN_DIR), ('Test', DRIVE_TEST_DIR)]:
            if d and os.path.exists(d):
                n = sum(1 for _ in Path(d).rglob('*') if _.is_file())
                print(f"   ✅ {name:5s}: {d}  ({n} file)")
            else:
                print(f"   ❌ {name:5s}: Tidak ditemukan → {d}")
        if DRIVE_VALID_DIR and os.path.exists(DRIVE_VALID_DIR):
            n = sum(1 for _ in Path(DRIVE_VALID_DIR).rglob('*') if _.is_file())
            print(f"   ✅ Valid: {DRIVE_VALID_DIR}  ({n} file)")
    else:
        if os.path.exists(DRIVE_SINGLE_DIR):
            n = sum(1 for _ in Path(DRIVE_SINGLE_DIR).rglob('*') if _.is_file())
            print(f"   ✅ Dataset folder: {DRIVE_SINGLE_DIR}  ({n} file total)")
        else:
            print(f"   ❌ Folder tidak ditemukan: {DRIVE_SINGLE_DIR}")
else:
    print(f"ℹ️  DATASET_SOURCE = '{DATASET_SOURCE}' → cell ini dilewati.")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 7 — Siapkan & Resolve Path Dataset
# ═══════════════════════════════════════════════════
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp', '.JPG', '.JPEG', '.PNG'}


def _count_images(folder: Path) -> int:
    return sum(1 for f in folder.rglob('*') if f.suffix in IMG_EXTS)


def _split_single_folder(source_dir: Path,
                          test_ratio: float,
                          valid_ratio: float,
                          split_base: Path = Path('/content/dataset_split')):
    '''
    Split sebuah folder (dengan subfolder kelas) ke train / val / test
    menggunakan sklearn train_test_split.
    Returns: (train_dir, val_dir_or_None, test_dir)
    '''
    train_dir = split_base / 'train'
    test_dir  = split_base / 'test'
    val_dir   = split_base / 'val' if valid_ratio > 0 else None

    if split_base.exists() and _count_images(split_base) > 0:
        print("ℹ️  Folder split sudah ada, digunakan langsung.")
        return str(train_dir), str(val_dir) if val_dir else None, str(test_dir)

    classes = sorted([d.name for d in source_dir.iterdir() if d.is_dir()])
    print(f"   Kelas ditemukan: {classes}")

    for cls in classes:
        imgs = sorted([f for f in (source_dir / cls).iterdir() if f.suffix in IMG_EXTS])
        if not imgs:
            print(f"   ⚠️  [{cls}] Tidak ada gambar, dilewati.")
            continue

        train_imgs, test_imgs = train_test_split(
            imgs, test_size=test_ratio, random_state=42, shuffle=True)

        val_imgs = []
        if valid_ratio > 0:
            val_from_train = valid_ratio / (1.0 - test_ratio)
            train_imgs, val_imgs = train_test_split(
                train_imgs, test_size=val_from_train, random_state=42)

        for img in train_imgs:
            dst = train_dir / cls / img.name
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img, dst)
        for img in test_imgs:
            dst = test_dir / cls / img.name
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img, dst)
        if val_imgs:
            for img in val_imgs:
                dst = val_dir / cls / img.name
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(img, dst)

        print(f"   [{cls}] Total:{len(imgs)}  Train:{len(train_imgs)}"
              + (f"  Val:{len(val_imgs)}" if val_imgs else "")
              + f"  Test:{len(test_imgs)}")

    return str(train_dir), str(val_dir) if val_dir else None, str(test_dir)


# ── Resolve path berdasarkan config ─────────────────────────
TRAIN_DIR = None
VAL_DIR   = None
TEST_DIR  = None

if DATASET_SOURCE == 'drive':
    if DRIVE_MODE == 'split_folders':
        TRAIN_DIR = DRIVE_TRAIN_DIR
        TEST_DIR  = DRIVE_TEST_DIR
        VAL_DIR   = DRIVE_VALID_DIR if DRIVE_VALID_DIR and os.path.exists(DRIVE_VALID_DIR) else None
    else:
        print(f"🔪 Melakukan train/val/test split dari: {DRIVE_SINGLE_DIR}")
        TRAIN_DIR, VAL_DIR, TEST_DIR = _split_single_folder(
            Path(DRIVE_SINGLE_DIR),
            test_ratio=SPLIT_TEST_RATIO,
            valid_ratio=SPLIT_VALID_RATIO,
        )

elif DATASET_SOURCE == 'kaggle':
    kaggle_base = Path(KAGGLE_UNZIP_DIR)
    train_cands = list(kaggle_base.rglob('train'))
    test_cands  = list(kaggle_base.rglob('test')) + list(kaggle_base.rglob('valid'))
    val_cands   = list(kaggle_base.rglob('val')) + list(kaggle_base.rglob('valid'))

    if train_cands and test_cands:
        TRAIN_DIR = str(next(d for d in train_cands if d.is_dir()))
        TEST_DIR  = str(next(d for d in test_cands  if d.is_dir()))
        VAL_DIR   = str(next(d for d in val_cands   if d.is_dir())) if val_cands else None
        print("✅ Auto-detected split folders dari Kaggle dataset.")
    else:
        print("⚠️  Struktur split tidak ditemukan → melakukan split manual.")
        TRAIN_DIR, VAL_DIR, TEST_DIR = _split_single_folder(
            kaggle_base,
            test_ratio=SPLIT_TEST_RATIO,
            valid_ratio=SPLIT_VALID_RATIO,
        )

# ── Verifikasi ──────────────────────────────────────────────
print("\n📁 Path Dataset:")
for name, d in [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR)]:
    if d:
        n = _count_images(Path(d)) if os.path.exists(d) else 0
        status = '✅' if n > 0 else '❌'
        print(f"   {status} {name:5s}: {d}  ({n} gambar)")

assert TRAIN_DIR and os.path.exists(TRAIN_DIR), "TRAIN_DIR tidak valid!"
assert TEST_DIR  and os.path.exists(TEST_DIR),  "TEST_DIR tidak valid!"
print("\n✅ Path dataset telah diverifikasi.")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 8 — Eksplorasi Dataset
# ═══════════════════════════════════════════════════
def explore_dataset(train_dir, test_dir, val_dir=None, output_dir=OUTPUT_DIR):
    '''Statistik jumlah gambar per kelas + visualisasi.'''
    split_dirs = {'Train': train_dir, 'Test': test_dir}
    if val_dir and os.path.exists(val_dir):
        split_dirs['Val'] = val_dir

    classes = sorted([d.name for d in Path(train_dir).iterdir() if d.is_dir()])
    print("=" * 70)
    print("📊 EKSPLORASI DATASET")
    print("=" * 70)
    print(f"\n🏷️  Kelas ({len(classes)} kelas):")
    for i, c in enumerate(classes):
        print(f"   [{i}] {c}")

    header = f"{'Split':<8}" + "".join(f"{c[:14]:<15}" for c in classes) + f"{'TOTAL':>8}"
    print(f"\n{header}")
    print("-" * len(header))

    stats = {}
    for sname, sdir in split_dirs.items():
        counts = []
        for cls in classes:
            p = Path(sdir) / cls
            n = len([f for f in p.iterdir() if f.suffix in IMG_EXTS]) if p.exists() else 0
            counts.append(n)
        total = sum(counts)
        stats[sname] = {'counts': counts, 'total': total}
        row = f"{sname:<8}" + "".join(f"{c:<15}" for c in counts) + f"{total:>8}"
        print(row)

    n_splits = len(split_dirs)
    fig, axes = plt.subplots(1, n_splits, figsize=(6 * n_splits, 5))
    if n_splits == 1:
        axes = [axes]
    colors = plt.cm.Set2(np.linspace(0, 1, len(classes)))

    for ax, (sname, sdir) in zip(axes, split_dirs.items()):
        counts = stats[sname]['counts']
        bars = ax.bar(classes, counts, color=colors, edgecolor='white', linewidth=0.8)
        ax.set_title(f'{sname} Set  (n={stats[sname]["total"]})', fontweight='bold')
        ax.set_ylabel('Jumlah Gambar')
        ax.tick_params(axis='x', rotation=40, labelsize=9)
        ax.grid(axis='y', alpha=0.3)
        for b, n in zip(bars, counts):
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5,
                    str(n), ha='center', va='bottom', fontsize=9)

    plt.suptitle('Distribusi Kelas per Split', fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    n_cols = min(len(classes), 4)
    n_rows = (len(classes) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = np.array(axes).flatten()

    for i, cls in enumerate(classes):
        cls_path = Path(train_dir) / cls
        imgs = [f for f in cls_path.iterdir() if f.suffix in IMG_EXTS]
        if imgs:
            img = Image.open(random.choice(imgs)).convert('RGB')
            axes[i].imshow(img)
        axes[i].set_title(cls, fontsize=9, fontweight='bold')
        axes[i].axis('off')
    for j in range(len(classes), len(axes)):
        axes[j].axis('off')

    plt.suptitle('Sampel Gambar per Kelas (Training Set)',
                 fontweight='bold', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()

    return classes


CLASS_NAMES = explore_dataset(TRAIN_DIR, TEST_DIR, VAL_DIR)
NUM_CLASSES = len(CLASS_NAMES)
print(f"\n✅ NUM_CLASSES = {NUM_CLASSES}  → {CLASS_NAMES}")

## ⚙️ KONFIGURASI HYPERPARAMETER
> Ubah nilai berikut sesuai kebutuhan sebelum menjalankan training.

| Variabel | Default | Keterangan |
|----------|---------|-----------|
| `NUM_EPOCHS` | 30 | Total epoch training |
| `BATCH_SIZE` | 32 | Kurangi ke 16 jika OOM |
| `LEARNING_RATE` | 1e-4 | LR untuk classifier head (Fase 1) |
| `LR_BACKBONE` | 1e-5 | LR saat fine-tuning backbone (Fase 2) |
| `FREEZE_EPOCHS` | 5 | Epoch fase 1: backbone dibekukan |
| `UNFREEZE_LAYERS` | 60 | Jumlah layer backbone yang di-unfreeze (-1 = semua) |
| `SCHEDULER_TYPE` | `'cosine'` | `'cosine'` / `'reduce_on_plateau'` / `'step'` |
| `USE_EARLY_STOPPING` | True | Aktifkan early stopping |
| `EARLY_STOP_PATIENCE` | 10 | Jumlah epoch tanpa peningkatan sebelum stop |
| `MIXED_PRECISION` | True | Percepat training di GPU T4/A100 dengan float16 |

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 9 — Konfigurasi Hyperparameter
# ═══════════════════════════════════════════════════
# ── Gambar ─────────────────────────────────────────
IMAGE_SIZE     = 224          # DenseNet121 standard input

# ── Training ───────────────────────────────────────
NUM_EPOCHS     = 30
BATCH_SIZE     = 32           # Turunkan ke 16 jika GPU OOM

# ── Learning Rate ──────────────────────────────────
LEARNING_RATE  = 1e-4         # LR untuk classifier head (Fase 1)
LR_BACKBONE    = 1e-5         # LR saat fine-tuning backbone (Fase 2)
WEIGHT_DECAY   = 1e-4         # L2 regularization (AdamW)
GRAD_CLIP_NORM = 1.0          # Gradient clipping (norm)

# ── Fine-tuning Strategy ───────────────────────────
FREEZE_EPOCHS    = 5          # Fase 1: backbone dibekukan selama N epoch
                               # Set 0 untuk langsung fine-tune semua layer
UNFREEZE_LAYERS  = 60         # Jumlah layer (Keras layer, bukan parameter)
                               # backbone yang di-unfreeze dari belakang.
                               # -1 = unfreeze semua layer backbone

# ── Regularization ─────────────────────────────────
DROPOUT_RATE     = 0.5        # Dropout pada classifier head
LABEL_SMOOTHING  = 0.1        # Label smoothing untuk CategoricalCrossentropy

# ── Scheduler ──────────────────────────────────────
SCHEDULER_TYPE = 'cosine'     # 'cosine' | 'reduce_on_plateau' | 'step'

COSINE_T_MAX   = NUM_EPOCHS   # Cosine annealing: periode (epoch)
COSINE_ETA_MIN = 1e-7         # Cosine annealing: LR minimum

PLATEAU_PATIENCE = 5          # ReduceLROnPlateau: patience (epoch)
PLATEAU_FACTOR   = 0.5        # ReduceLROnPlateau: faktor penurunan LR
PLATEAU_MIN_LR   = 1e-7       # ReduceLROnPlateau: LR minimum

STEP_SIZE      = 10           # StepDecay: setiap N epoch LR dikalikan GAMMA
STEP_GAMMA     = 0.1          # StepDecay: faktor perkalian

# ── Early Stopping ─────────────────────────────────
USE_EARLY_STOPPING   = True
EARLY_STOP_PATIENCE  = 10     # Stop jika val_acc tidak naik selama N epoch

# ── Checkpoint ─────────────────────────────────────
SAVE_EVERY_N_EPOCHS  = 5      # Simpan checkpoint arsip periodic (0 = skip)
CHECKPOINT_NAME      = 'densenet121_corn_disease'

# ── Precision ────────────────────────────────────────
MIXED_PRECISION = True        # float16 mixed precision (mempercepat di GPU modern)

if MIXED_PRECISION and len(tf.config.list_physical_devices('GPU')) > 0:
    keras.mixed_precision.set_global_policy('mixed_float16')
    print("✅ Mixed precision (mixed_float16) diaktifkan.")
else:
    keras.mixed_precision.set_global_policy('float32')
    print("ℹ️  Mixed precision tidak diaktifkan (CPU terdeteksi atau dinonaktifkan).")

# ── Print summary ──────────────────────────────────
print("=" * 55)
print("⚙️  RINGKASAN HYPERPARAMETER")
print("=" * 55)
for k, v in {
    "Image Size"         : f"{IMAGE_SIZE}×{IMAGE_SIZE}",
    "Epochs"             : NUM_EPOCHS,
    "Batch Size"         : BATCH_SIZE,
    "LR (head)"          : LEARNING_RATE,
    "LR (backbone)"      : LR_BACKBONE,
    "Weight Decay"       : WEIGHT_DECAY,
    "Dropout"            : DROPOUT_RATE,
    "Label Smoothing"    : LABEL_SMOOTHING,
    "Freeze Epochs"      : FREEZE_EPOCHS,
    "Unfreeze Layers"    : UNFREEZE_LAYERS,
    "Scheduler"          : SCHEDULER_TYPE,
    "Early Stop Patience": EARLY_STOP_PATIENCE,
    "Save Every N Epoch" : SAVE_EVERY_N_EPOCHS,
    "Mixed Precision"    : MIXED_PRECISION,
}.items():
    print(f"   {k:<22}: {v}")
print("=" * 55)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 10 — tf.data Pipeline (Dataset & Augmentasi)
# ═══════════════════════════════════════════════════
# ── Load dataset dari folder ────────────────────────────────
train_ds_raw = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='categorical',
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
)
CLASS_NAMES = train_ds_raw.class_names
NUM_CLASSES = len(CLASS_NAMES)

if VAL_DIR and os.path.exists(VAL_DIR):
    val_ds_raw = keras.utils.image_dataset_from_directory(
        VAL_DIR, labels='inferred', label_mode='categorical',
        image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE,
        shuffle=False,
    )
    print(f"✅ Val set dari folder: {VAL_DIR}")
else:
    # Auto-split 15% dari train sebagai val
    val_batches = int(0.15 * tf.data.experimental.cardinality(train_ds_raw).numpy())
    val_ds_raw   = train_ds_raw.take(val_batches)
    train_ds_raw = train_ds_raw.skip(val_batches)
    print(f"✅ Val set: 15% split dari training ({val_batches} batch)")

test_ds_raw = keras.utils.image_dataset_from_directory(
    TEST_DIR, labels='inferred', label_mode='categorical',
    image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE,
    shuffle=False,
)

# ── Augmentasi (dibungkus sebagai layer, dipakai di dalam model) ──
# Karena augmentasi ditempel di dalam arsitektur model (Cell 11),
# augmentasi HANYA aktif saat training=True dan otomatis nonaktif
# saat evaluasi/inferensi — tidak perlu logic tambahan.
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.15),
], name="data_augmentation")

# ── Optimasi pipeline: cache → shuffle → prefetch ───────────
train_ds = train_ds_raw.cache().shuffle(1000, seed=42).prefetch(AUTOTUNE)
val_ds   = val_ds_raw.cache().prefetch(AUTOTUNE)
test_ds  = test_ds_raw.cache().prefetch(AUTOTUNE)

n_train = tf.data.experimental.cardinality(train_ds).numpy() * BATCH_SIZE
n_val   = tf.data.experimental.cardinality(val_ds).numpy()   * BATCH_SIZE
n_test  = tf.data.experimental.cardinality(test_ds).numpy()  * BATCH_SIZE

print(f"\n📦 Dataset Summary:")
print(f"   Kelas    : {CLASS_NAMES}")
print(f"   Train    : ~{n_train} gambar  ({tf.data.experimental.cardinality(train_ds).numpy()} batch)")
print(f"   Val      : ~{n_val} gambar  ({tf.data.experimental.cardinality(val_ds).numpy()} batch)")
print(f"   Test     : ~{n_test} gambar  ({tf.data.experimental.cardinality(test_ds).numpy()} batch)")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 11 — Bangun Model DenseNet121 (Functional API)
# ═══════════════════════════════════════════════════
def build_densenet121(num_classes: int, image_size: int = IMAGE_SIZE,
                      dropout: float = 0.5):
    '''
    DenseNet121 dengan custom classifier head.
    Backbone ImageNet pretrained via keras.applications.

    Preprocessing (mode='torch': scale ke [0,1] lalu normalisasi
    dengan mean/std ImageNet) sudah dibakukan di dalam graph model,
    jadi saat inferensi cukup masukkan gambar RGB mentah (0-255).

    Arsitektur head:
        GAP → BN → Dropout → Dense(512, relu) → BN → Dropout → Dense(num_classes, softmax)
    '''
    base_model = keras.applications.DenseNet121(
        include_top=False,
        weights='imagenet',
        input_shape=(image_size, image_size, 3),
        pooling=None,
    )
    base_model._name = 'densenet121_backbone'   # nama tetap agar mudah di-retrieve saat resume
    base_model.trainable = False                # bekukan backbone (Fase 1)

    inputs = keras.Input(shape=(image_size, image_size, 3), name='input_image')
    x = data_augmentation(inputs)
    x = keras.applications.densenet.preprocess_input(x)
    # training=False → BatchNorm backbone SELALU inference-mode,
    # menjaga running statistics ImageNet tetap stabil walau nanti di-unfreeze.
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='head_bn1')(x)
    x = layers.Dropout(dropout, name='head_dropout1')(x)
    x = layers.Dense(512, activation='relu', name='head_dense')(x)
    x = layers.BatchNormalization(name='head_bn2')(x)
    x = layers.Dropout(dropout * 0.5, name='head_dropout2')(x)
    # dtype='float32' wajib di layer terakhir saat mixed precision aktif
    outputs = layers.Dense(num_classes, activation='softmax',
                           dtype='float32', name='predictions')(x)

    model = keras.Model(inputs, outputs, name='DenseNet121_CornDisease')
    return model, base_model


def unfreeze_backbone(base_model: keras.Model, n_layers: int = -1):
    '''
    Unfreeze N layer terakhir dari backbone DenseNet121.
    n_layers = -1 → unfreeze semua layer.
    Catatan: BatchNorm backbone tetap 'inference-mode' berkat
    training=False pada pemanggilan base_model() di build_densenet121(),
    sehingga aman untuk fine-tuning walau trainable=True.
    '''
    base_model.trainable = True
    total = len(base_model.layers)

    if n_layers == -1 or n_layers >= total:
        freeze_until = 0
    else:
        freeze_until = total - n_layers

    for i, layer in enumerate(base_model.layers):
        layer.trainable = (i >= freeze_until)

    n_trainable = sum(1 for l in base_model.layers if l.trainable)
    print(f"   🔓 Unfreeze {n_trainable}/{total} layer terakhir backbone.")


def model_summary(model: keras.Model, base_model: keras.Model = None):
    total     = model.count_params()
    trainable = sum(np.prod(w.shape) for w in model.trainable_weights)
    print(f"\n🏗️  Model: {model.name}")
    print(f"   Total params    : {int(total):>12,}")
    print(f"   Trainable params: {int(trainable):>12,}")
    print(f"   Frozen params   : {int(total - trainable):>12,}")


# Build model
model, base_model = build_densenet121(NUM_CLASSES, IMAGE_SIZE, DROPOUT_RATE)
model_summary(model, base_model)
model.summary(show_trainable=True)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 12 — Utilitas Training (Optimizer & Checkpoint)
# ═══════════════════════════════════════════════════
def make_optimizer(phase: str = 'head'):
    '''AdamW optimizer. LR berbeda tergantung fase training.'''
    lr = LEARNING_RATE if phase == 'head' else LR_BACKBONE
    return keras.optimizers.AdamW(
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        clipnorm=GRAD_CLIP_NORM,
    )


def compile_model(model, phase='head'):
    model.compile(
        optimizer=make_optimizer(phase),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=['accuracy'],
    )
    return model


def get_current_lr(model) -> float:
    lr = model.optimizer.learning_rate
    if hasattr(lr, 'numpy'):
        return float(lr.numpy())
    return float(keras.backend.get_value(lr))


def set_lr(model, new_lr: float):
    model.optimizer.learning_rate.assign(new_lr)


def compute_scheduled_lr(epoch: int, base_lr: float) -> float:
    '''Hitung LR untuk 'cosine' & 'step'. Untuk 'reduce_on_plateau'
    LR diatur manual di loop training berdasarkan patience counter.'''
    if SCHEDULER_TYPE == 'cosine':
        return COSINE_ETA_MIN + 0.5 * (base_lr - COSINE_ETA_MIN) * (
            1 + math.cos(math.pi * min(epoch, COSINE_T_MAX) / COSINE_T_MAX))
    elif SCHEDULER_TYPE == 'step':
        return base_lr * (STEP_GAMMA ** (epoch // STEP_SIZE))
    else:
        return None  # 'reduce_on_plateau' ditangani terpisah


# ── Checkpoint ──────────────────────────────────────────────
def save_checkpoint(model, epoch, best_val_acc, history, phase,
                    filename, is_best=False):
    path = Path(CHECKPOINT_DIR) / filename
    model.save(path)  # format .keras → menyimpan arsitektur + bobot + optimizer state

    meta = {
        'epoch'        : epoch,
        'best_val_acc' : best_val_acc,
        'history'      : history,
        'phase'        : phase,
        'class_names'  : CLASS_NAMES,
        'num_classes'  : NUM_CLASSES,
        'hyperparams': {
            'image_size'   : IMAGE_SIZE,
            'batch_size'   : BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'lr_backbone'  : LR_BACKBONE,
            'weight_decay' : WEIGHT_DECAY,
            'dropout_rate' : DROPOUT_RATE,
        },
        'saved_at': datetime.now().isoformat(),
    }
    meta_path = Path(CHECKPOINT_DIR) / f'{CHECKPOINT_NAME}_meta.json'
    meta_path.write_text(json.dumps(meta, indent=2))

    if is_best:
        best_path = Path(CHECKPOINT_DIR) / f'{CHECKPOINT_NAME}_best.keras'
        model.save(best_path)
        print(f"   💾 Best checkpoint → {best_path.name}")


def load_checkpoint(ckpt_path):
    '''Load model .keras (arsitektur+bobot+optimizer state) + metadata json.'''
    print(f"📂 Loading checkpoint: {ckpt_path}")
    model = keras.models.load_model(ckpt_path)
    base_model = model.get_layer('densenet121_backbone')

    meta_path = Path(CHECKPOINT_DIR) / f'{CHECKPOINT_NAME}_meta.json'
    meta = json.loads(meta_path.read_text())

    start_epoch  = meta['epoch'] + 1
    best_val_acc = meta['best_val_acc']
    history      = meta['history']
    phase        = meta['phase']

    print(f"✅ Resume dari epoch {start_epoch}  (fase: {phase})")
    print(f"   Best val acc sebelumnya : {best_val_acc*100:.2f}%")
    return model, base_model, start_epoch, best_val_acc, history, phase


def list_checkpoints(ckpt_dir=CHECKPOINT_DIR):
    ckpts = sorted(Path(ckpt_dir).glob('*.keras'))
    if not ckpts:
        print("ℹ️  Belum ada checkpoint.")
        return []
    print(f"📁 Checkpoints ({len(ckpts)} file):")
    for i, c in enumerate(ckpts):
        mb = sum(f.stat().st_size for f in [c]) / 1e6
        marker = '⭐' if 'best' in c.name else '  '
        print(f"   {marker}[{i}] {c.name}  ({mb:.1f} MB)")
    return ckpts

list_checkpoints()

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 13 — Loop Training
# ═══════════════════════════════════════════════════
def train_model(model, base_model, train_ds, val_ds,
                num_epochs=NUM_EPOCHS,
                start_epoch=0,
                best_val_acc=0.0,
                history=None,
                current_phase='head'):
    '''
    Training loop lengkap (epoch-by-epoch, kontrol manual):
    - Fase 1: backbone dibekukan (FREEZE_EPOCHS epoch)
    - Fase 2: backbone di-unfreeze (fine-tuning), LR lebih kecil
    - LR scheduling manual per-epoch (cosine / step / reduce_on_plateau)
    - Checkpoint saving (periodic + best) menggunakan format .keras
    - Early stopping
    - Live epoch log
    '''
    if history is None:
        history = {'train_loss': [], 'val_loss': [],
                   'train_acc': [],  'val_acc': [],
                   'lr': []}

    if not model.optimizer:  # belum pernah di-compile (training baru)
        compile_model(model, current_phase)

    early_stop_counter = 0
    plateau_counter    = 0
    base_lr            = LEARNING_RATE if current_phase == 'head' else LR_BACKBONE

    print("=" * 72)
    print("🚀 MULAI TRAINING — DenseNet121 Corn Leaf Disease Detector")
    print(f"   Epochs    : {start_epoch+1} → {num_epochs}")
    print(f"   Fase awal : {'Fase 2 (Fine-tuning)' if current_phase=='finetune' else 'Fase 1 (Head only)'}")
    print("=" * 72)

    t0_total = time.time()

    for epoch in range(start_epoch, num_epochs):
        t0_epoch = time.time()

        # ── Fase switch ──────────────────────────────────────
        if FREEZE_EPOCHS > 0 and epoch == FREEZE_EPOCHS and current_phase == 'head':
            print(f"\n{'─'*72}")
            print(f"🔓 Epoch {epoch+1}: Switch ke Fase 2 — Fine-tuning backbone ...")
            unfreeze_backbone(base_model, UNFREEZE_LAYERS)
            current_phase = 'finetune'
            base_lr = LR_BACKBONE
            compile_model(model, current_phase)   # wajib recompile setelah ubah .trainable
            model_summary(model, base_model)
            print(f"{'─'*72}\n")

        # ── LR scheduling (selain reduce_on_plateau) ─────────
        if SCHEDULER_TYPE in ('cosine', 'step'):
            epoch_in_phase = epoch - (FREEZE_EPOCHS if current_phase == 'finetune' else 0)
            new_lr = compute_scheduled_lr(max(epoch_in_phase, 0), base_lr)
            set_lr(model, new_lr)

        # ── Train 1 epoch (Keras native fit, tetap pakai progress bar bawaan) ──
        hist = model.fit(train_ds, validation_data=val_ds,
                         initial_epoch=epoch, epochs=epoch + 1, verbose=1)

        tr_loss = hist.history['loss'][0]
        tr_acc  = hist.history['accuracy'][0]
        vl_loss = hist.history['val_loss'][0]
        vl_acc  = hist.history['val_accuracy'][0]
        cur_lr  = get_current_lr(model)

        # ── reduce_on_plateau (manual) ────────────────────────
        if SCHEDULER_TYPE == 'reduce_on_plateau':
            if vl_acc > best_val_acc:
                plateau_counter = 0
            else:
                plateau_counter += 1
                if plateau_counter >= PLATEAU_PATIENCE:
                    new_lr = max(cur_lr * PLATEAU_FACTOR, PLATEAU_MIN_LR)
                    set_lr(model, new_lr)
                    cur_lr = new_lr
                    plateau_counter = 0
                    print(f"   📉 ReduceLROnPlateau → LR baru: {new_lr:.2e}")

        # ── History ──────────────────────────────────────────
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)
        history['lr'].append(cur_lr)

        # ── Best & early stopping ─────────────────────────────
        is_best = vl_acc > best_val_acc
        if is_best:
            best_val_acc       = vl_acc
            early_stop_counter = 0
        else:
            early_stop_counter += 1

        # ── Checkpoint ───────────────────────────────────────
        if SAVE_EVERY_N_EPOCHS > 0 and (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
            fname = f'{CHECKPOINT_NAME}_epoch{epoch+1:03d}.keras'
            save_checkpoint(model, epoch, best_val_acc, history, current_phase, fname)
            print(f"   💾 Checkpoint: {fname}")

        # Selalu simpan 'latest' agar resume selalu bisa dari epoch paling akhir
        save_checkpoint(model, epoch, best_val_acc, history, current_phase,
                        f'{CHECKPOINT_NAME}_latest.keras')

        if is_best:
            save_checkpoint(model, epoch, best_val_acc, history, current_phase,
                            f'{CHECKPOINT_NAME}_best.keras', is_best=True)

        # ── Epoch log ─────────────────────────────────────────
        dt   = time.time() - t0_epoch
        star = '⭐' if is_best else '  '
        print(
            f"{star} [{epoch+1:3d}/{num_epochs}] "
            f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
            f"| val_loss={vl_loss:.4f}  val_acc={vl_acc:.4f}  "
            f"| lr={cur_lr:.1e}  | {dt:.1f}s"
        )

        # ── Early stopping ────────────────────────────────────
        if USE_EARLY_STOPPING and early_stop_counter >= EARLY_STOP_PATIENCE:
            print(f"\n⏹️  Early stopping! "
                  f"Val acc tidak naik selama {EARLY_STOP_PATIENCE} epoch.")
            break

    elapsed = time.time() - t0_total
    print(f"\n{'='*72}")
    print(f"✅ Training selesai!")
    print(f"   Waktu total  : {elapsed/60:.1f} menit")
    print(f"   Best val acc : {best_val_acc*100:.2f}%")
    print(f"{'='*72}")

    return model, history, best_val_acc, current_phase

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 14 — Mulai / Lanjutkan Training
# ═══════════════════════════════════════════════════
# ── Opsi Resume ────────────────────────────────────
RESUME_TRAINING   = True    # True = coba load checkpoint terbaru
CHECKPOINT_TO_USE = None    # None = auto (latest → best)
                             # Atau: 'densenet121_corn_disease_epoch010.keras'

# ── Inisialisasi ────────────────────────────────────
start_epoch    = 0
best_val_acc   = 0.0
history        = None
current_phase  = 'head'

if RESUME_TRAINING:
    if CHECKPOINT_TO_USE:
        ckpt_path = str(Path(CHECKPOINT_DIR) / CHECKPOINT_TO_USE)
    else:
        latest_path = Path(CHECKPOINT_DIR) / f'{CHECKPOINT_NAME}_latest.keras'
        best_path   = Path(CHECKPOINT_DIR) / f'{CHECKPOINT_NAME}_best.keras'
        ckpt_path = str(latest_path) if latest_path.exists() else (
            str(best_path) if best_path.exists() else None)

    if ckpt_path and os.path.exists(ckpt_path):
        model, base_model, start_epoch, best_val_acc, history, current_phase = \
            load_checkpoint(ckpt_path)
    else:
        print("ℹ️  Checkpoint tidak ditemukan → mulai dari awal.")
        model, base_model = build_densenet121(NUM_CLASSES, IMAGE_SIZE, DROPOUT_RATE)
else:
    model, base_model = build_densenet121(NUM_CLASSES, IMAGE_SIZE, DROPOUT_RATE)

model_summary(model, base_model)

# ── Jalankan training ────────────────────────────────
model, history, best_val_acc, current_phase = train_model(
    model=model,
    base_model=base_model,
    train_ds=train_ds,
    val_ds=val_ds,
    num_epochs=NUM_EPOCHS,
    start_epoch=start_epoch,
    best_val_acc=best_val_acc,
    history=history,
    current_phase=current_phase,
)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 15 — Visualisasi Training History
# ═══════════════════════════════════════════════════
def plot_history(history, freeze_epochs=FREEZE_EPOCHS, out_dir=OUTPUT_DIR):
    epochs = range(1, len(history['train_loss']) + 1)
    best_e = int(np.argmax(history['val_acc'])) + 1

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Training History — DenseNet121 Corn Leaf Disease (TensorFlow)',
                 fontsize=14, fontweight='bold')

    ax = axes[0]
    ax.plot(epochs, history['train_loss'], 'b-o', label='Train', ms=3, lw=1.5)
    ax.plot(epochs, history['val_loss'],   'r-o', label='Val',   ms=3, lw=1.5)
    ax.set_title('Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    ax.plot(epochs, [a*100 for a in history['train_acc']], 'b-o', label='Train', ms=3, lw=1.5)
    ax.plot(epochs, [a*100 for a in history['val_acc']],   'r-o', label='Val',   ms=3, lw=1.5)
    ax.axvline(best_e, color='gold', ls='--', lw=1.5,
               label=f'Best (ep {best_e}, {max(history["val_acc"])*100:.2f}%)')
    ax.set_title('Accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[2]
    ax.plot(epochs, history['lr'], 'g-o', ms=3, lw=1.5)
    ax.set_title('Learning Rate Schedule')
    ax.set_xlabel('Epoch'); ax.set_ylabel('LR')
    ax.set_yscale('log'); ax.grid(alpha=0.3)

    if freeze_epochs > 0 and freeze_epochs < len(list(epochs)):
        for ax in axes:
            ylims = ax.get_ylim()
            ax.axvline(freeze_epochs, color='purple', ls=':', lw=1.5, alpha=0.7)
            ax.text(freeze_epochs + 0.2, ylims[0] + (ylims[1]-ylims[0])*0.05,
                    'Unfreeze', color='purple', fontsize=8, rotation=90)

    plt.tight_layout()
    plt.savefig(f'{out_dir}/training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n📊 Training Summary:")
    print(f"   Best Val Loss : {min(history['val_loss']):.4f}  (epoch {int(np.argmin(history['val_loss']))+1})")
    print(f"   Best Val Acc  : {max(history['val_acc'])*100:.2f}%  (epoch {best_e})")
    print(f"   Final Train   : loss={history['train_loss'][-1]:.4f}  acc={history['train_acc'][-1]*100:.2f}%")
    print(f"   Final Val     : loss={history['val_loss'][-1]:.4f}   acc={history['val_acc'][-1]*100:.2f}%")


plot_history(history)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 16 — Evaluasi Model pada Test Set
# ═══════════════════════════════════════════════════
def evaluate(model, test_ds, class_names, out_dir=OUTPUT_DIR):
    '''
    Evaluasi model pada test set.
    Output: Confusion Matrix (count & %) + Classification Report.
    Otomatis load best checkpoint jika ada.
    '''
    best_path = Path(CHECKPOINT_DIR) / f'{CHECKPOINT_NAME}_best.keras'
    if best_path.exists():
        model = keras.models.load_model(best_path)
        print(f"✅ Best model dimuat: {best_path.name}")
    else:
        print("⚠️  Best checkpoint tidak ditemukan, menggunakan model di memory.")

    print("\n🔍 Evaluasi pada Test Set ...")
    all_labels, all_preds, all_probs = [], [], []

    for batch_imgs, batch_labels in test_ds:
        probs = model.predict(batch_imgs, verbose=0)
        preds = np.argmax(probs, axis=1)
        labels = np.argmax(batch_labels.numpy(), axis=1)

        all_preds.extend(preds)
        all_labels.extend(labels)
        all_probs.extend(probs)

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    test_acc = accuracy_score(all_labels, all_preds)
    test_f1  = f1_score(all_labels, all_preds, average='weighted')
    print(f"\n{'='*70}")
    print(f"🎯 Test Accuracy : {test_acc*100:.4f}%")
    print(f"🎯 Test F1-Score : {test_f1*100:.4f}%  (weighted)")
    print(f"{'='*70}")

    print("\n📋 Classification Report:")
    report = classification_report(all_labels, all_preds,
                                   target_names=class_names, digits=4)
    print(report)

    rpt_path = Path(out_dir) / 'classification_report.txt'
    rpt_path.write_text(
        f"Test Accuracy : {test_acc*100:.4f}%\n"
        f"Test F1-Score : {test_f1*100:.4f}%\n\n"
        f"Classification Report:\n{'='*70}\n{report}"
    )
    print(f"📄 Report disimpan ke: {rpt_path}")

    cm      = confusion_matrix(all_labels, all_preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(max(14, 4*len(class_names)), 6))
    kw = dict(xticklabels=class_names, yticklabels=class_names,
              linewidths=0.5, cmap='Blues')

    sns.heatmap(cm,      annot=True, fmt='d',    ax=axes[0], **kw)
    axes[0].set_title(f'Confusion Matrix — Jumlah\nTest Acc: {test_acc*100:.2f}%',
                      fontweight='bold')
    axes[0].set_xlabel('Prediksi'); axes[0].set_ylabel('Aktual')
    axes[0].tick_params(axis='x', rotation=40)

    sns.heatmap(cm_norm, annot=True, fmt='.2%', ax=axes[1],
                vmin=0, vmax=1, **kw)
    axes[1].set_title('Confusion Matrix — Persentase', fontweight='bold')
    axes[1].set_xlabel('Prediksi'); axes[1].set_ylabel('Aktual')
    axes[1].tick_params(axis='x', rotation=40)

    plt.tight_layout()
    cm_path = Path(out_dir) / 'confusion_matrix.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"📊 Confusion matrix disimpan ke: {cm_path}")

    print(f"\n📊 Per-class Accuracy:")
    for i, cls in enumerate(class_names):
        acc_i  = cm_norm[i, i] * 100
        bar    = '█' * int(acc_i / 5)
        print(f"   {cls:<35} {bar:<20} {acc_i:.2f}%")

    return {
        'test_accuracy'   : test_acc,
        'test_f1'         : test_f1,
        'confusion_matrix': cm,
        'predictions'     : all_preds,
        'labels'          : all_labels,
        'probabilities'   : all_probs,
        'model'           : model,
    }


eval_results = evaluate(model, test_ds, CLASS_NAMES)
model = eval_results['model']   # gunakan best model untuk cell selanjutnya

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 17 — Simpan Model Akhir
# ═══════════════════════════════════════════════════
SAVE_TO_DRIVE_FINAL = True        # Simpan copy ke Google Drive
DRIVE_MODEL_DIR     = '/content/drive/MyDrive/models/corn_disease'
CONVERT_TO_TFLITE   = True        # Export tambahan ke TFLite (untuk mobile/edge)
FINAL_MODEL_NAME    = (
    f"densenet121_corn_{datetime.now().strftime('%Y%m%d_%H%M')}"
    f"_acc{eval_results['test_accuracy']*100:.1f}"
)


def export_model(model, class_names, out_dir, model_name,
                 drive_dir=None, eval_res=None, to_tflite=True):
    '''Export model ke berbagai format.'''
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    meta = {
        'arch'         : 'DenseNet121',
        'framework'    : 'TensorFlow/Keras',
        'class_names'  : class_names,
        'num_classes'  : len(class_names),
        'image_size'   : IMAGE_SIZE,
        'preprocessing': "keras.applications.densenet.preprocess_input (mode='torch')",
        'hyperparams'  : {
            'dropout_rate' : DROPOUT_RATE,
            'learning_rate': LEARNING_RATE,
            'lr_backbone'  : LR_BACKBONE,
            'weight_decay' : WEIGHT_DECAY,
            'batch_size'   : BATCH_SIZE,
        },
        'test_accuracy': float(eval_res['test_accuracy']) if eval_res else None,
        'test_f1'      : float(eval_res['test_f1'])       if eval_res else None,
        'trained_at'   : datetime.now().isoformat(),
    }

    saved_files = []

    # 1. Full model (.keras) — arsitektur + bobot + optimizer state
    keras_path = out / f'{model_name}.keras'
    model.save(keras_path)
    print(f"✅ Model (.keras)  : {keras_path}")
    saved_files.append(keras_path)

    # 2. Weights only
    w_path = out / f'{model_name}.weights.h5'
    model.save_weights(w_path)
    print(f"✅ Weights only    : {w_path}")
    saved_files.append(w_path)

    # 3. Metadata JSON
    json_path = out / f'{model_name}_metadata.json'
    json_path.write_text(json.dumps(meta, indent=2))
    print(f"✅ Metadata JSON   : {json_path}")
    saved_files.append(json_path)

    # 4. TFLite (opsional, untuk deployment mobile/edge)
    if to_tflite:
        try:
            converter = tf.lite.TFLiteConverter.from_keras_model(model)
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
            tflite_model = converter.convert()
            tflite_path = out / f'{model_name}.tflite'
            tflite_path.write_bytes(tflite_model)
            print(f"✅ TFLite          : {tflite_path}")
            saved_files.append(tflite_path)
        except Exception as e:
            print(f"⚠️  Konversi TFLite gagal (opsional): {e}")

    # 5. Copy ke Google Drive
    if drive_dir:
        try:
            Path(drive_dir).mkdir(parents=True, exist_ok=True)
            for f in saved_files:
                dst = Path(drive_dir) / f.name
                shutil.copy2(f, dst)
                print(f"☁️  → Drive: {dst}")
        except Exception as e:
            print(f"⚠️  Gagal simpan ke Drive: {e}")

    print(f"\n{'='*60}")
    print(f"✅ Model berhasil diekspor!")
    print(f"   Nama       : {model_name}")
    if meta['test_accuracy']:
        print(f"   Test Acc   : {meta['test_accuracy']*100:.2f}%")
    print(f"   Kelas      : {class_names}")
    print(f"{'='*60}")


export_model(
    model, CLASS_NAMES,
    out_dir=OUTPUT_DIR,
    model_name=FINAL_MODEL_NAME,
    drive_dir=DRIVE_MODEL_DIR if SAVE_TO_DRIVE_FINAL else None,
    eval_res=eval_results,
    to_tflite=CONVERT_TO_TFLITE,
)

## ⚙️ KONFIGURASI INFERENSI
> Atur variabel di bawah, lalu jalankan **Cell 19**

| Variabel | Keterangan |
|----------|-----------|
| `INF_MODE` | `'single'` → 1 gambar, `'batch'` → banyak gambar |
| `INF_IMAGE` | Path gambar tunggal |
| `INF_BATCH` | List path gambar untuk batch |
| `INF_FOLDER` | Folder berisi gambar (jika batch) |
| `UPLOAD_IMAGE` | `True` → upload dari PC lokal |
| `USE_CURRENT_MODEL` | `True` → gunakan model di memory (tanpa load file) |

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 18 — Konfigurasi Inferensi
# ═══════════════════════════════════════════════════
INF_MODE    = 'single'     # 'single' | 'batch'

INF_IMAGE   = '/content/test_image.jpg'          # untuk mode 'single'
INF_BATCH   = [                                   # untuk mode 'batch' (list)
    # '/content/img1.jpg',
    # '/content/img2.jpg',
]
INF_FOLDER  = ''    # Kosongkan atau isi folder path untuk mode 'batch'

UPLOAD_IMAGE = False    # True = muncul dialog upload file

USE_CURRENT_MODEL = True    # True = gunakan model di memory
INF_MODEL_PATH = ''          # Path file .keras jika USE_CURRENT_MODEL = False
                              # Contoh: f'{OUTPUT_DIR}/{FINAL_MODEL_NAME}.keras'

CONFIDENCE_THRESHOLD = 0.5   # Prediksi < threshold → tampilkan peringatan

print("✅ Konfigurasi inferensi tersimpan.")
print(f"   Mode     : {INF_MODE}")
print(f"   Upload   : {UPLOAD_IMAGE}")
print(f"   Model    : {'Memory' if USE_CURRENT_MODEL else INF_MODEL_PATH}")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 19 — Inferensi Gambar Baru
# ═══════════════════════════════════════════════════
# Catatan: Preprocessing (resize + normalisasi ImageNet) sudah
# dibakukan di dalam graph model (Cell 11), jadi cukup masukkan
# gambar RGB mentah berukuran (IMAGE_SIZE, IMAGE_SIZE, 3) rentang 0-255.

def load_and_preprocess_image(img_path: str, image_size: int = IMAGE_SIZE):
    img = keras.utils.load_img(img_path, target_size=(image_size, image_size))
    arr = keras.utils.img_to_array(img)  # float32, range 0-255
    return img, arr


def predict_one(inf_model, img_path: str, class_names: list,
                show: bool = True, out_dir: str = OUTPUT_DIR):
    '''Prediksi satu gambar dan tampilkan hasilnya.'''
    img, arr = load_and_preprocess_image(img_path)
    batch    = np.expand_dims(arr, axis=0)
    probs    = inf_model.predict(batch, verbose=0)[0]

    pred_idx   = int(probs.argmax())
    pred_class = class_names[pred_idx]
    confidence = float(probs[pred_idx])

    if show:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))

        axes[0].imshow(img)
        color = '#27ae60' if confidence >= 0.7 else \
                '#f39c12' if confidence >= CONFIDENCE_THRESHOLD else '#e74c3c'
        axes[0].set_title(
            f"Prediksi: {pred_class}\nKonfidensі: {confidence*100:.2f}%",
            color=color, fontweight='bold', fontsize=13
        )
        axes[0].axis('off')

        sorted_idx  = np.argsort(probs)[::-1]
        sorted_cls  = [class_names[i] for i in sorted_idx]
        sorted_prob = probs[sorted_idx] * 100
        bar_colors  = ['#27ae60' if c == pred_class else '#3498db' for c in sorted_cls]

        bars = axes[1].barh(sorted_cls, sorted_prob, color=bar_colors, edgecolor='white')
        axes[1].set_xlim(0, 105)
        axes[1].set_xlabel('Probabilitas (%)')
        axes[1].set_title('Distribusi Probabilitas', fontweight='bold')
        axes[1].axvline(CONFIDENCE_THRESHOLD * 100, color='orange',
                        ls='--', lw=1.2, label=f'Threshold {CONFIDENCE_THRESHOLD*100:.0f}%')
        axes[1].legend(fontsize=9)
        axes[1].grid(axis='x', alpha=0.3)

        for bar, pct in zip(bars, sorted_prob):
            axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                         f'{pct:.2f}%', va='center', fontsize=10)

        plt.tight_layout()
        plt.savefig(f'{out_dir}/inference_single.png', dpi=150, bbox_inches='tight')
        plt.show()

    warn = ' ⚠️ (konfidensі rendah!)' if confidence < CONFIDENCE_THRESHOLD else ''
    print(f"\n🌽 Hasil Deteksi Penyakit Daun Jagung:")
    print(f"   File        : {Path(img_path).name}")
    print(f"   Penyakit    : {pred_class}{warn}")
    print(f"   Konfidensī  : {confidence*100:.2f}%")
    print(f"\n   Semua Probabilitas:")
    for i in np.argsort(probs)[::-1]:
        bar = '█' * int(probs[i] * 25)
        print(f"   {class_names[i]:<35} {bar:<26} {probs[i]*100:.2f}%")

    return {'class': pred_class, 'confidence': confidence,
            'all_probs': {class_names[i]: float(probs[i])
                          for i in range(len(class_names))}}


def predict_batch_imgs(inf_model, img_paths: list, class_names: list,
                       out_dir: str = OUTPUT_DIR):
    '''Prediksi banyak gambar dan tampilkan grid hasil.'''
    n = len(img_paths)
    n_cols = min(4, n)
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = np.array(axes).flatten()

    results = []
    print(f"🔍 Inferensi {n} gambar ...")

    # Batch semua gambar sekaligus untuk efisiensi
    imgs_pil, arrs = [], []
    for p in img_paths:
        img, arr = load_and_preprocess_image(p)
        imgs_pil.append(img)
        arrs.append(arr)
    batch = np.stack(arrs, axis=0)
    all_probs = inf_model.predict(batch, verbose=0)

    for i, (img_path, img, probs) in enumerate(zip(img_paths, imgs_pil, all_probs)):
        pred_idx = int(probs.argmax())
        conf     = float(probs[pred_idx])
        pred_cls = class_names[pred_idx]

        color = '#27ae60' if conf >= 0.7 else \
                '#f39c12' if conf >= CONFIDENCE_THRESHOLD else '#e74c3c'
        axes[i].imshow(img)
        axes[i].set_title(f"{pred_cls}\n{conf*100:.1f}%",
                          color=color, fontsize=9, fontweight='bold')
        axes[i].axis('off')

        results.append({'path': str(img_path), 'class': pred_cls, 'confidence': conf})
        marker = '⚠️' if conf < CONFIDENCE_THRESHOLD else '  '
        print(f"   {marker}[{i+1:2d}] {Path(img_path).name:<30}"
              f"→ {pred_cls:<35} ({conf*100:.1f}%)")

    for j in range(n, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'Hasil Deteksi — {n} Gambar', fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{out_dir}/inference_batch.png', dpi=150, bbox_inches='tight')
    plt.show()

    counts = Counter(r['class'] for r in results)
    print(f"\n📊 Ringkasan Prediksi Batch ({n} gambar):")
    for cls, cnt in counts.most_common():
        pct = cnt / n * 100
        print(f"   {cls:<35} : {cnt:3d} gambar  ({pct:.1f}%)")

    return results


# ══════════════════════════════════════════════════
# ── Jalankan Inferensi ─────────────────────────────
# ══════════════════════════════════════════════════

# 1. Upload gambar (opsional)
if UPLOAD_IMAGE:
    from google.colab import files
    print("📂 Upload gambar:")
    uploaded = files.upload()
    up_paths = []
    for fname, fdata in uploaded.items():
        p = f'/content/{fname}'
        Path(p).write_bytes(fdata)
        up_paths.append(p)
    print(f"✅ {len(up_paths)} gambar diupload: {up_paths}")
    if INF_MODE == 'single':
        INF_IMAGE = up_paths[0]
    else:
        INF_BATCH = up_paths

# 2. Load model
if USE_CURRENT_MODEL:
    best_path = Path(CHECKPOINT_DIR) / f'{CHECKPOINT_NAME}_best.keras'
    if best_path.exists():
        inf_model = keras.models.load_model(best_path)
        print("✅ Best model dimuat untuk inferensi.")
    else:
        inf_model = model
    inf_classes = CLASS_NAMES
else:
    assert INF_MODEL_PATH and os.path.exists(INF_MODEL_PATH), \
        f"File model tidak ditemukan: {INF_MODEL_PATH}"
    inf_model = keras.models.load_model(INF_MODEL_PATH)
    inf_classes = CLASS_NAMES
    print(f"✅ Model dimuat dari: {INF_MODEL_PATH}")

# 3. Kumpulkan gambar untuk batch
if INF_MODE == 'batch':
    if INF_FOLDER and os.path.exists(INF_FOLDER):
        INF_BATCH = sorted([
            str(p) for p in Path(INF_FOLDER).rglob('*')
            if p.suffix in IMG_EXTS
        ])
        print(f"📁 Ditemukan {len(INF_BATCH)} gambar di folder: {INF_FOLDER}")
    INF_BATCH = [p for p in INF_BATCH if os.path.exists(p)]

# 4. Jalankan prediksi
if INF_MODE == 'single':
    if os.path.exists(INF_IMAGE):
        result = predict_one(inf_model, INF_IMAGE, inf_classes)
    else:
        print(f"❌ File tidak ditemukan: {INF_IMAGE}")
        print("   Atur UPLOAD_IMAGE = True atau perbaiki INF_IMAGE.")
elif INF_MODE == 'batch':
    if INF_BATCH:
        results = predict_batch_imgs(inf_model, INF_BATCH, inf_classes)
    else:
        print("❌ Tidak ada gambar untuk inferensi batch.")
        print("   Isi INF_BATCH atau INF_FOLDER, atau set UPLOAD_IMAGE = True.")